In [ ]:
!pip install --upgrade monai

In [ ]:
import os
import shutil
os.environ['CUDA_LAUNCH_BLOCKING'] = '1' 
import pandas as pd
from glob import glob
from PIL import Image
from matplotlib import pyplot as plt
import nibabel as nib
from ipywidgets import interact
import numpy as np
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import torch.optim as optim
from torch.amp import autocast, GradScaler
from torch.utils.checkpoint import checkpoint
from collections import OrderedDict
from sklearn.model_selection import train_test_split
from monai.transforms import (
    LoadImage,
    AsDiscrete,
    AsDiscreted,
    Compose,
    LoadImaged,
    Orientationd,
    Spacingd,
    CropForegroundd,
    SpatialPadd,
    ScaleIntensityRanged,
    # RandFlipd,
    # RandAffined,
    EnsureChannelFirstd,
    # CenterSpatialCropd,
    # ResampleToMatch,
    Spacing,
    Invertd,
    FgBgToIndicesd,
    EnsureTyped,
    RandCropByPosNegLabeld,
    NormalizeIntensityd,
    RandFlipd,
    # RandRotated, 
    # RandZoomd,
    # RandGaussianNoised,
    # NormalizeIntensityd,
    RandScaleIntensityd,
    RandShiftIntensityd,
    # RandAdjustContrastd,
    # Rand3DElasticd,
    MapTransform,
    Lambdad,
    SaveImaged
)
from monai.data import (
    PersistentDataset,
    CacheDataset, 
    Dataset, 
    ThreadDataLoader, 
    DataLoader, 
    decollate_batch,
    list_data_collate,
    pad_list_data_collate,
    set_track_meta
)
from monai.handlers.utils import from_engine
from monai.networks.nets import UNet, SwinUNETR
from monai.networks.layers import Norm, Act
from monai.metrics import DiceMetric
from monai.losses import DiceLoss, DiceCELoss, DiceFocalLoss
from monai.inferers import sliding_window_inference

from monai.config import print_config

# from monai.apps import download_and_extract

import torch.utils.checkpoint as cp
torch.backends.cudnn.benchmark = True

import time
from copy import deepcopy

print_config()

In [ ]:
print(torch.__version__)

In [ ]:
# !nvidia-smi
from monai.apps import download_url
from monai.bundle import download

**LOAD DATA & VISUALIZE**

In [ ]:
# Lấy đường dẫn data
def load_data_paths(base_path):
    train_imgs = sorted(glob(os.path.join(base_path, "train", "volume", "*.nii")))
    train_masks  = sorted(glob(os.path.join(base_path, "train", "segmentation", "*.nii")))
    test_imgs   = sorted(glob(os.path.join(base_path, "test", "*.nii")))

    return train_imgs, train_masks, test_imgs

In [ ]:
ROOT_PATH = '/kaggle/input/aio2025liverseg'

# check root_path
for root, dirs, files in os.walk(ROOT_PATH):
    print(root, len(files))
    break

train_imgs, train_masks, test_imgs = load_data_paths(ROOT_PATH)

print(f'Size train dataset images: {len(train_imgs)}')
print(f'Size train dataset masks: {len(train_masks)}')
print(f'Size test dataset images: {len(test_imgs)}')

In [ ]:
# Chọn ảnh, mask đầu tiên trong tập train
sample_img_path = train_imgs[0]
sample_mask_path = train_masks[0]
print("File:", sample_img_path)
print("File:", sample_mask_path)

# Load ảnh
img = nib.load(sample_img_path)
mask = nib.load(sample_mask_path)
data_img = img.get_fdata()
data_mask = mask.get_fdata()
print("Shape (X,Y,Z):", data_img.shape)
print("Shape (X,Y,Z):", data_mask.shape)

In [ ]:
slice_idx = 53

fig, axes = plt.subplots(1, 2, figsize=(12, 6))  # 1 hàng, 2 cột

# Ảnh gốc
axes[0].imshow(data_img[:, :, slice_idx].T, cmap="gray", origin="lower")
axes[0].set_title(f"CT Slice {slice_idx}")
axes[0].axis("off")

# Mask
axes[1].imshow(data_mask[:, :, slice_idx].T, cmap="gray", origin="lower")
axes[1].set_title(f"Mask Slice {slice_idx}")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
slice_idx = 53

fig, ax = plt.subplots(figsize=(4,4))

# Ảnh gốc (volume)
ax.imshow(data_img[:, :, slice_idx].T,
          cmap="gray", origin="lower")

# Mask segmentation — dùng alpha để nhìn xuyên
ax.imshow(data_mask[:, :, slice_idx].T,
          cmap="autumn", origin="lower", alpha=0.4)

ax.set_title(f"Slice {slice_idx}")
ax.axis("off")
plt.show()

In [ ]:
def view_slice(idx):
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))  # 1 hàng, 2 cột
    
    # Ảnh gốc
    axes[0].imshow(data_img[:, :, idx].T, cmap="gray", origin="lower")
    axes[0].set_title(f"CT Slice {idx}")
    axes[0].axis("off")

    # Mask
    axes[1].imshow(data_mask[:, :, idx].T, cmap="gray", origin="lower")
    axes[1].set_title(f"Mask Slice {idx}")
    axes[1].axis("off")
    
    plt.tight_layout()
    plt.show()

interact(view_slice, idx=(0, data_img.shape[2]-1))

**DATA AUGMENTATION & TRANSFORM**

**cài đặt global cho track meta**

In [ ]:
# set trace back invert transform
set_track_meta(True)

**khởi tạo data transform cho tập train và val**

In [ ]:
# Transform cho tập train
train_transform = Compose(
    [
        LoadImaged(keys=["image", "label"]),
        # Thêm bước này để đảm bảo định dạng tensor là (C, D, H, W)
        EnsureChannelFirstd(keys=["image", "label"]),
        # Chuẩn hóa hướng ảnh
        Orientationd(keys=["image", "label"], axcodes="RAS", labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))),
        # Resample về kích thước 1mm isotropic
        Spacingd(keys=["image", "label"], pixdim=(1.5, 1.5, 1.5), mode=("bilinear", "nearest")),
        # Cắt hoặc đệm ảnh để đảm bảo kích thước đồng nhất
        CropForegroundd(keys=["image", "label"], source_key="image"),
        # Pad nếu vùng gan nhỏ
        # SpatialPadd(keys=["image", "label"], spatial_size=(128, 128, 64), mode="constant"), 
        SpatialPadd(keys=["image", "label"], spatial_size=(96, 96, 96), mode="constant"), 
        # Chuẩn hóa cường độ pixel
        ScaleIntensityRanged(keys=["image"], a_min=-100, a_max=200, b_min=0.0, b_max=1.0, clip=True),
        NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),
        EnsureTyped(keys=["image", "label"], track_meta=False),
        # # Sử dụng SpatialCropd để cắt ảnh về kích thước cố định, đảm bảo không bị OOM
        # CenterSpatialCropd(keys=["image", "label"], roi_size=(128, 128, 64)),
        # # Sau đó mới dùng SpatialPadd để đệm nếu cần
        # SpatialPadd(keys=["image", "label"], spatial_size=(128, 128, 64), mode="constant"),
    ]
)

aug_transform = Compose([
    RandCropByPosNegLabeld(
        keys=["image", "label"], label_key="label",
        # spatial_size=(128, 128, 64),
        spatial_size=(96, 96, 96),
        pos=3, neg=2, num_samples=5,
        image_key="image", image_threshold=0,
    ),
    RandFlipd(keys=["image", "label"], spatial_axis=[0], prob=0.5),
    RandScaleIntensityd(keys=["image"], factors=0.1, prob=0.5),
    RandShiftIntensityd(keys=["image"], offsets=0.1, prob=0.5),
    Lambdad(keys=["image", "label"], func=lambda x: torch.as_tensor(x) if isinstance(x, np.ndarray) else x),
    EnsureTyped(keys=["image", "label"], track_meta=False),
])
    

# Transform cho tập validation (không tăng cường dữ liệu)
val_transform = Compose(
    [
        LoadImaged(keys=["image", "label"], image_only=False),
        # Thêm bước này để đảm bảo định dạng tensor là (C, D, H, W)
        EnsureChannelFirstd(keys=["image", "label"]),
        Orientationd(keys=["image", "label"], axcodes="RAS", labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))),
        Spacingd(keys=["image", "label"], pixdim=(1.5, 1.5, 1.5), mode=("bilinear", "nearest")),
        CropForegroundd(keys=["image", "label"], source_key="image"),
        # SpatialPadd(keys=["image", "label"], spatial_size=(128, 128, 64), mode="constant"),
        SpatialPadd(keys=["image", "label"], spatial_size=(96, 96, 96), mode="constant"),
        ScaleIntensityRanged(keys=["image"], a_min=-100, a_max=200, b_min=0.0, b_max=1.0, clip=True),
        NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),
        EnsureTyped(keys=["image", "label"], track_meta=True), 
    ]
)

**khởi tạo dataloader cho tập train và val**

In [ ]:
# Chuyển đổi hai danh sách thành một danh sách các dictionary (key-value)
data_dicts = [{"image": img, "label": mask} for img, mask in zip(train_imgs, train_masks)]

# Chia 70/10 (train/val)
train_files, val_files = train_test_split(data_dicts, test_size=0.125, random_state=42, shuffle=True)
print(f"Train: {len(train_files)} samples")
print(f"Val:   {len(val_files)} samples")
 
# train_set = Dataset(data=train_files, transform=train_transform)
# val_set = Dataset(data=val_files, transform=val_transform)

# train_set = CacheDataset(data=train_files, transform=train_transform, cache_rate=1.0, num_workers=4)
# val_set = CacheDataset(data=val_files, transform=val_transform, cache_rate=1.0, num_workers=4)

train_set = PersistentDataset(data=train_files, transform=train_transform, cache_dir="./data_train_cache")
val_set = PersistentDataset(data=val_files, transform=val_transform, cache_dir="./data_val_cache")

train_loader = DataLoader(Dataset(train_set, aug_transform), batch_size=1, shuffle=True, num_workers=4, pin_memory=True, collate_fn=pad_list_data_collate)
val_loader = DataLoader(val_set, batch_size=1, shuffle=False, num_workers=4, pin_memory=True, collate_fn=pad_list_data_collate)

**Check data transform và đếm số tumor nhỏ** \
(bước check data transform sẽ tiện cho việc train vì sẽ cache transform trước ở bước này) 

In [ ]:
# data_iterator = iter(train_loader)
data_iterator = iter(train_set)
sample_batch = next(data_iterator)
sample_img = sample_batch["image"]
sample_mask = sample_batch["label"]

print(type(sample_img), sample_img.dtype, sample_img.shape)
print(type(sample_mask), sample_mask.dtype, sample_mask.shape)

In [ ]:
def check_tumor_size(label, tumor_value=2, min_voxel=500):
    """
    label: Tensor 3D hoặc 4D (C, D, H, W), thường sau EnsureChannelFirstd thì C=1
    tumor_value: giá trị voxel của tumor trong mask (ví dụ 2 nếu liver=1, tumor=2)
    min_voxel: ngưỡng số voxel để coi tumor là 'nhỏ'
    """
    # Chỉ lấy channel đầu (C=1)
    if label.ndim == 4:
        label = label[0]  
    
    # Đếm voxel thuộc tumor
    tumor_voxels = torch.sum(label == tumor_value).item()
    
    # Check nhỏ hay không
    if tumor_voxels < min_voxel:
        return True, tumor_voxels
    else:
        return False, tumor_voxels

In [ ]:
small_count, large_count = 0, 0
for batch in train_loader:
    for lbl in batch["label"]:
        is_small, vox = check_tumor_size(lbl, tumor_value=2, min_voxel=500)
        if vox > 0:  # chỉ tính patch có tumor
            if is_small:
                small_count += 1
            else:
                large_count += 1

print(f"Tumor nhỏ (<500 voxels): {small_count}")
print(f"Tumor lớn (>=500 voxels): {large_count}")

In [ ]:
sample = train_set[0]
print(type(sample["image"]), type(sample["label"]))

In [ ]:
batch = next(iter(train_loader))
for k, v in batch.items():
    print(f"{k}: {type(v)}")
    if isinstance(v, torch.Tensor):
        print("   shape:", v.shape, "dtype:", v.dtype)
    elif isinstance(v, list):
        print("   list of types:", [type(x) for x in v])

**TRAINING**

**Define evaluate function**

In [ ]:
# khởi tạo hàm post-process để đưa data sau predict về one-hot và lấy argmax
post_pred = Compose([AsDiscrete(argmax=True, to_onehot=3)])
post_label = Compose([AsDiscrete(to_onehot=3)])

In [ ]:
# ----- Evaluate function -----
def evaluate(model, loader, device, criterion, dice_metric):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:            
            val_inputs = batch["image"]
            val_labels = batch["label"]
            val_inputs, val_labels = val_inputs.to(device), val_labels.to(device)
            
            # roi_size = (128, 128, 64)
            roi_size = (96, 96, 96)
            sw_batch_size = 4

            # precdict theo cửa sổ trượt
            val_outputs = sliding_window_inference(val_inputs, roi_size, sw_batch_size, model, overlap=0.5, mode="gaussian")
            
            loss = criterion(val_outputs, val_labels)
            total_loss += loss.item()

            # tách batch thành các sample để post-process
            val_outputs_decollate = [post_pred(i) for i in decollate_batch(val_outputs)]
            val_labels_decollate = [post_label(i) for i in decollate_batch(val_labels)]
            
            # compute metric for current iteration
            dice_metric(y_pred=val_outputs_decollate, y=val_labels_decollate)
            
        # aggregate the final mean dice val result
        mean_dice = dice_metric.aggregate().item()
        # reset the status for next validation round
        dice_metric.reset()
        
        # aggregate the final mean loss val result
        mean_loss = total_loss / len(loader)
        
    return mean_loss, mean_dice

**--- define train function---**

In [ ]:
def train_model(model, train_loader, val_loader, device, epochs=50, lr=2e-4, patience=25):
    # criterion = DiceLoss(to_onehot_y=True, softmax=True)
    
    criterion = DiceCELoss(
        include_background=False,   # tính background nếu dataset balance thì để True
        to_onehot_y=True,          # tự one-hot label
        softmax=True,              # nếu dùng multi-class -> softmax; nếu binary thì sigmoid=True
        lambda_dice=1.0,           # hệ số dice
        lambda_ce=1.0,              # hệ số CE
        # weight=torch.tensor([0.05, 0.35, 0.6], device=device) # weight theo lambda_ce có thể tinh chỉnh nếu muốn
    )

    # criterion = DiceFocalLoss(
    #     include_background=False,   
    #     to_onehot_y=True,
    #     softmax=True,
    #     lambda_dice=1.0,
    #     lambda_focal=1.0,           
    #     # weight=torch.tensor([0.1, 0.3, 0.6], device=device),
    #     gamma=2.0,                  
    # )
    
    dice_metric = DiceMetric(include_background=False, reduction="mean")
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=8)

    scaler = GradScaler()
    start_epoch = 0
    best_dice = 0.0
    counter = 0
    

    checkpoint_path = "checkpoint.pth"
    
    if os.path.exists(checkpoint_path):
        print(f"Tải checkpoint từ {checkpoint_path}...")
        checkpoint = torch.load(checkpoint_path, map_location=device)

        state_dict_to_load = checkpoint['model_state_dict']
        
        # current_model khởi tạo trọng số từ model base hoặc model wrapper nếu có
        current_model = model.module if isinstance(model, nn.DataParallel) else model
        current_model.load_state_dict(state_dict_to_load)
        
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch']
        best_dice = checkpoint['best_dice']
        counter = checkpoint['counter']
        print(f"Đã tải checkpoint thành công. Tiếp tục huấn luyện từ epoch {start_epoch + 1} - best dice {best_dice}.")

    train_losses, val_losses, dice_scores = [], [], []
    
    for epoch in range(start_epoch + 1, epochs+1):
        start = time.time()
        # model base/ model wrapper
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            images = batch["image"]
            masks = batch["label"]
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()

            with autocast(device_type=device, dtype=torch.float16):
                preds = model(images)
                loss = criterion(preds, masks)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        val_loss, val_dice = evaluate(model, val_loader, device, criterion, dice_metric)
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        dice_scores.append(val_dice)

        scheduler.step(val_dice)

        print(f"Current learning rate: {optimizer.param_groups[0]['lr']}")
        print(f"Epoch [{epoch}/{epochs}] Loss: {train_loss:.5f} | Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.5f}")
            
        # Lưu checkpoint sau mỗi epoch
        current_model = model.module if isinstance(model, nn.DataParallel) else model
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': current_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_dice': best_dice,
            'counter': counter
        }
        torch.save(checkpoint, checkpoint_path)
        if val_dice > best_dice:
            best_dice = val_dice
            counter = 0
            torch.save(current_model.state_dict(), "best_model.pth")
        else:
            counter += 1
            if counter >= patience:
                print("Early stopping triggered")
                break
                
        print(f"Epoch {epoch} time:", time.time() - start)
        torch.cuda.empty_cache()

    if os.path.exists("best_model.pth"):
        current_model = model.module if isinstance(model, nn.DataParallel) else model
        current_model.load_state_dict(torch.load("best_model.pth", map_location=device))
    return model, train_losses, val_losses, dice_scores

**check RAM, CPU**

In [ ]:
import psutil
import os

print("\n--- Thông tin RAM ---")
# Lấy thông tin về bộ nhớ ảo (RAM)
mem = psutil.virtual_memory()

# Tổng RAM được cung cấp cho môi trường Notebook
total_ram_gb = mem.total / (1024**3)
print(f"Tổng RAM (Total): {total_ram_gb:.2f} GB")

# RAM đang sử dụng (Used)
used_ram_gb = mem.used / (1024**3)
print(f"RAM đang sử dụng: {used_ram_gb:.2f} GB")

# Phần trăm RAM đang sử dụng
print(f"Phần trăm sử dụng: {mem.percent}%")

print("--- Thông tin CPU ---")
# Số lượng lõi logic (thường là số được cung cấp cho Notebook)
num_cores_logical = os.cpu_count()
print(f"Số lượng lõi Logic: {num_cores_logical}")

# Số lượng lõi vật lý
num_cores_physical = psutil.cpu_count(logical=False)
print(f"Số lượng lõi Vật lý: {num_cores_physical}")

# Chi tiết tần suất CPU
cpu_freq = psutil.cpu_freq()
print(f"Tần suất CPU hiện tại: {cpu_freq.current/1000:.2f} GHz")
print(f"Tần suất Max CPU: {cpu_freq.max/1000:.2f} GHz")

**Download pretrained weight**

pretrained only weight

In [ ]:
# download("swin_unetr_btcv_segmentation", bundle_dir="./", progress=False)

In [ ]:
# state_dict1 = torch.load("swin_unetr_btcv_segmentation/models/model.pt", map_location="cpu")
# print(type(state_dict1), state_dict1.keys())

pretrained có thể dùng để fine-tune tiếp (epoch, best_acc, weight)

In [ ]:
# ---btcv feature-size 24---
url = "https://github.com/Project-MONAI/MONAI-extra-test-data/releases/download/0.8.1/swin_unetr.small_5000ep_f24_lr2e-4_pretrained.pt"
download_url(url, "./small_5000ep_f24_lr2e-4_pretrained.pt")

In [ ]:
state_dict1 = torch.load("small_5000ep_f24_lr2e-4_pretrained.pt", map_location="cpu", weights_only=False)
print(type(state_dict1), state_dict1.keys())

In [ ]:
# ---btcv feature-size 48---

# url = "https://github.com/Project-MONAI/MONAI-extra-test-data/releases/download/0.8.1/swin_unetr.base_5000ep_f48_lr2e-4_pretrained.pt"
# download_url(url, "./base_5000ep_f48_lr2e-4_pretrained.pt")

In [ ]:
# ---self-supervised learning weight---

# url = "https://github.com/Project-MONAI/MONAI-extra-test-data/releases/download/0.8.1/ssl_pretrained_weights.pth"
# download_url(url, "./ssl_pretrained_weights.pth")

In [ ]:
# state_dict = torch.load("ssl_pretrained_weights.pth", map_location="cpu")
# print(type(state_dict), state_dict.keys())

**TRAIN**

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(torch.cuda.is_available())          # True nếu có GPU
print(torch.cuda.device_count())          # Số lượng GPU
print(torch.cuda.get_device_name(0))      # Tên GPU đầu tiên
print(torch.cuda.memory_summary(device='cuda:0', abbreviated=False))

# model = UNet(
#     spatial_dims=3,                     # 3D
#     in_channels=1,                      # CT input 1 channel
#     out_channels=3,                     # 3 class: background, liver, tumor
#     channels=(16, 32, 64, 128, 256),    # số kênh mỗi layer (nhẹ hơn 32,64,128,256,512)
#     strides=(2, 2, 2, 2),               # downsampling
#     num_res_units=2,                    # số residual unit mỗi layer
#     norm=Norm.INSTANCE,                 # InstanceNorm tốt hơn với batch nhỏ
#     act=Act.PRELU,                      # activation
#     kernel_size=3,
#     up_kernel_size=3,         
#     dropout=0.1,              
# ).to(device)

model = SwinUNETR(
    in_channels=1,
    out_channels=3,
    feature_size=24,                     # số channels của feature map
    drop_rate=0.0,                       
    attn_drop_rate=0.0,
    dropout_path_rate=0.0,
    use_checkpoint=True,
).to(device)

checkpoint_base = torch.load("small_5000ep_f24_lr2e-4_pretrained.pt", map_location="cpu", weights_only=False)
weights = checkpoint_base["state_dict"]
filtered_weights = {k: v for k, v in weights.items() if not k.startswith("out.")}

missing, unexpected = model.load_state_dict(filtered_weights, strict=False)
print("Model loaded with:")
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

# if torch.cuda.device_count() > 1:
#     print(f"Sử dụng {torch.cuda.device_count()} GPU!")
    # model = nn.DataParallel(model)       # model wrapper by DataParallel

model, train_losses, val_losses, dice_scores = train_model(
    model, train_loader, val_loader, device, epochs=250
)

In [ ]:
# Plot training curves
def plot_training_curves(train_losses, val_losses, dice_scores):
    epochs = range(1, len(train_losses) + 1)

    plt.style.use("ggplot")
    plt.figure(figsize=(14,5))

    # Loss
    plt.subplot(1,2,1)
    plt.plot(epochs, train_losses, 'b-', label='Train Loss')
    plt.plot(epochs, val_losses, 'r-', label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Train & Validation Loss')
    plt.legend()
    plt.grid(True)

    # Metrics
    plt.subplot(1,2,2)
    plt.plot(epochs, dice_scores, 'g-', label='Dice Score')
    plt.xlabel('Epoch')
    plt.ylabel('Score')
    plt.title('Dice Scores')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

plot_training_curves(train_losses, val_losses, dice_scores)

# Check best model output with the input image and label

In [ ]:
def view_slice(idx, img, label, pred):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))  # 3 cột: ảnh, GT, pred

    # Ảnh CT gốc
    axes[0].imshow(img[:, :, idx].T, cmap="gray", origin="lower")
    axes[0].set_title(f"CT Slice {idx}")
    axes[0].axis("off")

    # Ground truth
    axes[1].imshow(label[:, :, idx].T, cmap="gray", origin="lower")
    axes[1].set_title("Ground Truth")
    axes[1].axis("off")

    # Prediction
    axes[2].imshow(pred[:, :, idx].T, cmap="gray", origin="lower")
    axes[2].set_title("Prediction")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
from ipywidgets import interact, fixed
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.eval()

with torch.no_grad():
    for i, val_data in enumerate(val_loader):
        # roi_size = (128, 128, 64)
        roi_size = (96, 96, 96)
        sw_batch_size = 4
    
        val_inputs = val_data["image"].to(device)
        val_labels = val_data["label"].to(device)
    
        # Dự đoán
        val_outputs = sliding_window_inference(
            val_inputs, roi_size, sw_batch_size, model, overlap=0.5, mode="gaussian"
        )
    
        # Chuyển về numpy
        img_np = val_inputs[0,0].cpu().numpy()
        label_np = val_labels[0,0].cpu().numpy()
        pred_np = val_outputs[0,0].detach().cpu().numpy()   # lấy kênh đầu tiên (nếu muốn argmax thì thêm bước dưới)
    
        # Nếu muốn mask segmentation thì argmax theo kênh
        pred_np = torch.argmax(val_outputs, dim=1)[0].cpu().numpy()
    
        # Interactive slider
        interact(
            view_slice,
            idx=(0, pred_np.shape[2]-1),
            img=fixed(img_np),
            label=fixed(label_np),
            pred=fixed(pred_np)
        )
        
        if i == 2:
            break

# Evaluation on original image spacings

**khởi tạo transform cho tập val**

In [ ]:
# có thể khởi tạo với các transform khác cho tập val để đánh giá
val_orig_transform = Compose(
    [
        LoadImaged(keys=["image", "label"], image_only=False),
        # Thêm bước này để đảm bảo định dạng tensor là (C, D, H, W)
        EnsureChannelFirstd(keys=["image", "label"]),
        Orientationd(keys=["image", "label"], axcodes="RAS", labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))),
        Spacingd(keys=["image", "label"], pixdim=(1.5, 1.5, 1.5), mode=("bilinear", "nearest")),
        CropForegroundd(keys=["image", "label"], source_key="image"),
        # SpatialPadd(keys=["image", "label"], spatial_size=(128, 128, 64), mode="constant"),
        SpatialPadd(keys=["image", "label"], spatial_size=(96, 96, 96), mode="constant"),
        ScaleIntensityRanged(keys=["image"], a_min=-100, a_max=200, b_min=0.0, b_max=1.0, clip=True),
        NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),
        EnsureTyped(keys=["image", "label"], track_meta=True),
    ]
)

**khởi tạo invert transform cho tập val**

In [ ]:
post_transforms = Compose(
    [
        # Invert trên image
        Invertd(
            # transform=val_orig_transform,       # transform mới bên trên
            transform=val_transform,             # transform ban đầu
            keys="pred",                         # key mới sau invert
            orig_keys="image",                   # key transform gốc
            meta_keys="pred_meta_dict",          # key lưu trữ data sau invert       vd: {"pred_meta_dict": "pred": []}
            orig_meta_keys="image_meta_dict",    # key gốc lưu trữ data trước invert  vd: {"image_meta_dict: "image:" []"}
            meta_key_postfix="meta_dict",        # hậu tố của key
            nearest_interp=False,
            device="cpu",
        ),
        
        # Invert trên Label
        Invertd(
            # transform=val_orig_transform,
            transform=val_transform,
            keys="label",
            orig_keys="label",
            meta_keys="label_meta_dict",
            orig_meta_keys="label_meta_dict",
            nearest_interp=True,
            device="cpu",
        ),
        
        AsDiscreted(keys="pred", argmax=True, to_onehot=3), 
        AsDiscreted(keys="label", to_onehot=3),
    ]
)

**Đánh giá trên không gian gần gốc (invert được 1 phần)**

In [ ]:
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.eval()

dice_metric = DiceMetric(include_background=False, reduction="mean")
dice_metric_per_class = DiceMetric(include_background=False, reduction="none")

with torch.no_grad():
    # val_loader dùng PersistanceDataset 
    for val_data in val_loader:
        val_inputs = val_data["image"].to(device)
        # roi_size = (128, 128, 64)
        roi_size = (96, 96, 96)
        sw_batch_size = 4
        pred = sliding_window_inference(val_inputs, roi_size, sw_batch_size, model, overlap=0.5, mode="gaussian")

        # đưa về CPU để tránh OOM khi invert
        val_data["pred"] = pred.cpu()
        val_data["image"] = val_data["image"].cpu()
        val_data["label"] = val_data["label"].cpu()

        print(val_data.keys())
        
        val_data = [post_transforms(i) for i in decollate_batch(val_data)]
        val_outputs, val_labels = from_engine(["pred", "label"])(val_data)
        
        val_outputs = [v.to(device) for v in val_outputs]
        val_labels = [v.to(device) for v in val_labels]

        for i, (pred, label) in enumerate(zip(val_outputs, val_labels)):
            print(f"[{i}] pred shape: {pred.shape}, label shape: {label.shape}")

        # compute metrics
        dice_metric(y_pred=val_outputs, y=val_labels)
        # dice_metric_batch(y_pred=val_outputs, y=val_labels)
        dice_metric_per_class(y_pred=val_outputs, y=val_labels)

    # mean dice
    metric_org = dice_metric.aggregate().item()
    dice_metric.reset()

    # # per batch
    # metric_org_per_batch = dice_metric_batch.aggregate().cpu().numpy()
    # metric_org_per_batch.reset()

    # metric_tc = metric_org_per_batch[0].item()
    # metric_wt = metric_org_per_batch[1].item()
    # metric_et = metric_org_per_batch[2].item()
    
    # per-class dice
    metric_org_per_class = dice_metric_per_class.aggregate().cpu().numpy()
    dice_metric_per_class.reset()
    


print("Metric on original image spacing (mean):", metric_org)
# print("Dice tc:", metric_tc)
# print("Dice wt:", metric_wt)
# print("Dice et:", metric_et)
print("Dice per class:", metric_org_per_class)   # [dice_liver, dice_tumor]
print("Mean Dice:", metric_org_per_class.mean())

In [ ]:
# compare with 3D-Unet

# Metric on original image spacing (mean): 0.7631502151489258
# Dice per class: [[0.93800616 0.63789195]
#  [0.95667094 0.4184739 ]
#  [0.9387809  0.8156605 ]
#  [0.93726534 0.47063062]
#  [0.95922863 0.70430106]
#  [0.9335147  0.6054484 ]
#  [0.91930795 0.5981308 ]
#  [0.91749    0.42330322]
#  [0.9270389  0.56653875]
#  [0.9465168  0.6488052 ]]
# Mean Dice: 0.7631502

# Test + Encode

In [ ]:
def rle_encode(mask: np.ndarray) -> str:
    """
    Run-length encode a 2D binary mask (1 for foreground, 0 for background)
    Empty mask -> '1 0'.
    """
    assert mask.ndim == 2, "rle_encode expects a 2D mask"
    pixels = mask.astype(np.uint8).flatten(order='F')  # column-major
    if pixels.max() == 0:
        return "1 0"
    # Pad with zeros at both ends to catch transitions cleanly
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(map(str, runs))

**khởi tạo transform, invert transform, dataset, dataloader cho tập test**

In [ ]:
# Chuyển đổi hai danh sách thành một danh sách các dictionary
test_data = [{"image": image} for image in test_imgs]

# khởi tạo transform
test_org_transforms = Compose(
    [
        LoadImaged(keys="image", image_only=False),
        EnsureChannelFirstd(keys="image"),
        Orientationd(keys=["image"], axcodes="RAS", labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))),
        Spacingd(keys=["image"], pixdim=(1.5, 1.5, 1.5), mode="bilinear"),
        CropForegroundd(keys=["image"], source_key="image", allow_smaller=True),
        # SpatialPadd(keys=["image"], spatial_size=(128, 128, 64), mode="constant"),
        SpatialPadd(keys=["image"], spatial_size=(96, 96, 96), mode="constant"),
        ScaleIntensityRanged(keys=["image"], a_min=-100, a_max=200, b_min=0.0, b_max=1.0, clip=True),
        NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),
        EnsureTyped(keys=["image"], track_meta=True),
    ]
)

# khởi tạo dataset (có thể dùng CacheDataset nếu nhiều Ram)
test_org_ds = Dataset(data=test_data, transform=test_org_transforms)

sample = test_org_ds[0]
print("🔑 Keys trong sample:", sample.keys())
print("📂 File gốc:", sample["image_meta_dict"]["filename_or_obj"])

# khởi tạo dataloader
test_org_loader = DataLoader(test_org_ds, batch_size=1, shuffle=False, num_workers=2, pin_memory=True, collate_fn=pad_list_data_collate)

# khai báo nơi lưu mask
mask_output_dir = "./mask_predictions"

if os.path.exists(mask_output_dir): # Xóa nếu đã tồn tại để tránh lỗi
    shutil.rmtree(mask_output_dir)
if not os.path.exists(mask_output_dir):
    os.makedirs(mask_output_dir)
    print(f"Đã tạo thư mục lưu mask 3D: {mask_output_dir}")

# khởi tạo invert transform
save_mask_transforms = Compose(
    [
        Invertd(
            keys="pred",
            transform=test_org_transforms,
            orig_keys="image",
            meta_keys="pred_meta_dict",
            orig_meta_keys="image_meta_dict",
            meta_key_postfix="meta_dict",
            nearest_interp=False,
            device="cpu",
        ),
        EnsureTyped(keys="pred"),
        AsDiscreted(keys="pred", argmax=True),
        
        # LƯU FILE 3D
        SaveImaged(
            keys="pred", 
            meta_keys="pred_meta_dict", 
            output_dir=mask_output_dir, 
            output_postfix="seg", 
            resample=False 
        ),
    ]
)

**khởi tạo model**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = UNet(
#     spatial_dims=3,          
#     in_channels=1,            
#     out_channels=3,           
#     channels=(16, 32, 64, 128, 256),   
#     strides=(2, 2, 2, 2),        
#     num_res_units=2,          
#     norm=Norm.INSTANCE,       
#     act=Act.PRELU,            
#     kernel_size=3,
#     up_kernel_size=3,         
#     dropout=0.1,              
# ).to(device)

model = SwinUNETR(
    in_channels=1,
    out_channels=3,
    feature_size=24,
    drop_rate=0.0,
    attn_drop_rate=0.0,
    dropout_path_rate=0.0,
    use_checkpoint=True,
).to(device)


# Tải trọng số
model.load_state_dict(torch.load("best_model.pth", map_location=device))

# if torch.cuda.device_count() > 1:
#     print(f"Sử dụng {torch.cuda.device_count()} GPU!")
#     model = nn.DataParallel(model)

model.eval()

**kiểm tra tập test và lấy thông tin image test**

In [ ]:
from monai.transforms import LoadImage

data = test_imgs[0]
loader = LoadImage(image_only=False)
img, meta = loader(data)
print(f"🧩 Image shape:", img.shape)
print("🧩 Metadata keys:", meta.keys())

In [ ]:
print(meta['filename_or_obj'])
img_id = meta['filename_or_obj']
img_id = img_id.split("/")[-1].split(".")[0].split("-")[-1]
print(img_id)
print(f'{int(img_id)}')

**predict**

In [ ]:
import csv
import os

output_csv = "submission1.csv"
mask_output_dir = "mask_predictions"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# mở file CSV để ghi
with open(output_csv, mode="w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "rle"])  # header

    with torch.no_grad():
        for i, batch in enumerate(test_org_loader):
            input_imgs = batch["image"].to(device)

            # roi_size = (128, 128, 64)
            roi_size = (96, 96, 96)
            sw_batch_size = 4

            # predict
            pred = sliding_window_inference(input_imgs, roi_size, sw_batch_size, model, overlap=0.5, mode="gaussian")
            batch["pred"] = pred.cpu()
            batch["image"] = batch["image"].cpu()

            # xử lý post-transforms
            data_post_processed = [save_mask_transforms(data) for data in decollate_batch(batch)]

            for data in data_post_processed:
                meta = data["image_meta_dict"]
                if meta is None:
                    print(f"⚠️ Warning: image_meta_dict trống, bỏ qua sample {data}")
                    continue

                image_id = os.path.basename(meta["filename_or_obj"])
                image_id = image_id.split("/")[-1].split(".")[0].split("-")[-1]
                print(image_id)

                print(f"🔍 pred shape before squeeze: {data['pred'].shape}")
                pred_tensor = data["pred"].squeeze().numpy().astype(np.uint8)
                print(f"🔍 pred shape after squeeze: {data['pred'].shape}")
                

                if pred_tensor.ndim != 3:
                    print(f"Lỗi: Tensor dự đoán không phải 3D. Bỏ qua {image_id}")
                    continue

                binary_mask = np.where(pred_tensor > 0, 1, 0).astype(np.uint8)
                num_slices = binary_mask.shape[2]

                # Ghi trực tiếp từng slice vào CSV
                for slice_idx in range(num_slices):
                    slice_mask = binary_mask[:, :, slice_idx]
                    rle = rle_encode(slice_mask)
                    writer.writerow([f"cell_{int(image_id)}_{slice_idx}", rle])
            
            # flush buffer mỗi batch lớn để giảm RAM
            f.flush()

print(f"✅ Đã tạo file submission RLE: {output_csv}")

**visualize mask**

In [ ]:
img_test = nib.load("/kaggle/input/aio2025liverseg/test/volume-082.nii")
mask_test = nib.load("/kaggle/working/mask_predictions/volume-082/volume-082_seg.nii.gz")
print(img_test.shape)
print(mask_test.shape)

data_img_test = img_test.get_fdata()
data_mask_test = mask_test.get_fdata()

def view_slice_test(idx):
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))  # 1 hàng, 2 cột
    
    # Ảnh gốc
    axes[0].imshow(data_img_test[:, :, idx].T, cmap="gray", origin="lower")
    axes[0].set_title(f"CT Slice {idx}")
    axes[0].axis("off")

    # Mask
    axes[1].imshow(data_mask_test[:, :, idx].T, cmap="gray", origin="lower")
    axes[1].set_title(f"Mask Slice {idx}")
    axes[1].axis("off")
    
    plt.tight_layout()
    plt.show()

interact(view_slice_test, idx=(0, data_mask_test.shape[2]-1))

In [ ]:
import monai
print(monai.__version__)